# 03 · Milestone 2: Model integration and baseline results

**Goal:** condition pre-trained Stable Diffusion 1.5 on CLIP ViT-L/14 text embeddings, run a baseline,
and tune **classifier-free guidance** and the **noise schedule** (at least 3 values each).

| Milestone 2 requirement | Where in this notebook |
|---|---|
| Integrate a text encoder with a diffusion model | §1 explicit denoising loop: CLIP → U-Net cross-attention → VAE |
| Run baseline conditional generation | §2 baseline: DPM++, 30 steps, w = 7.5 |
| Tune classifier-free guidance (≥ 3 values) | §3 w = 1, 3, 5, 7.5, 10, 15, 20 |
| Tune the noise schedule (≥ 3 values) | §4 sampler (DDPM, DDIM, PNDM, DPM++) and timestep spacing (leading, linspace, trailing) |
| Record the first set of generated images | §6 generation log + §7 figures, saved to `outputs/milestone2/` |

**Design.** The same 10 prompts (5 COCO captions + 5 handwritten) with one fixed seed per prompt are used
in every run, and each sweep changes **one** setting from the baseline. Any difference between images
therefore comes from that setting alone.

> **Runtime:** needs a GPU. On a Colab T4 the full notebook takes about 30–40 minutes
> (≈ 150 images at 5–10 s each, plus model downloads).

In [ ]:
# --- Setup: works in Google Colab and locally ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HiralR2931/Generative_Project_DeepLearning.git"
BRANCH = "yvonne/param-sweeps"        # change to "main" once the pull request is merged
REPO_DIR = "Generative_Project_DeepLearning"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.chdir("/content")
    current = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
    if current != BRANCH:             # missing, empty, or on another branch -> fresh clone
        !rm -rf {REPO_DIR}
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")                    # run from repo root locally

sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())
assert Path("src/sweeps.py").exists(), f"src/sweeps.py not found: is branch '{BRANCH}' pushed to GitHub?"


def download_outputs():
    # Colab deletes files when the session ends: zip outputs/milestone2 and download it.
    import shutil
    if not IN_COLAB:
        print("Results are saved in", Path("outputs/milestone2").resolve())
        return
    from google.colab import files
    zip_path = shutil.make_archive("/content/milestone2_outputs", "zip",
                                   root_dir="outputs", base_dir="milestone2")
    files.download(zip_path)

In [ ]:
# [settings] everything you may want to change is here
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from src import config as C

REQUIRE_GPU = True
MODEL_ID = C.SD_MODEL                 # Stable Diffusion 1.5
CLIP_ID = C.CLIP_MODEL                # CLIP ViT-L/14 (also used for the CLIP score)
SIZE = C.IMAGE_SIZE                   # 512 x 512

BASELINE = dict(scheduler="dpm++", guidance_scale=7.5, num_steps=30)    # same as Milestone 1
GUIDANCE_VALUES = [1, 3, 5, 7.5, 10, 15, 20]
SAMPLERS = ["ddpm", "ddim", "pndm", "dpm++"]
SPACINGS = ["leading", "linspace", "trailing"]
RUN_STEPS_SWEEP = True                # optional extra: number of sampling steps
STEP_VALUES = [10, 25, 30, 50]

OUT = C.OUT_DIR / "milestone2"        # images, generation log, figures
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.generate import load_pipeline, generate
from src.denoise import encode_text, manual_generate
from src.sweeps import (SweepRunner, m2_prompts, make_scheduler, run_info, add_clip_scores, summarize,
                        make_grid, make_gallery, plot_guidance_curve, plot_setting_bars, plot_noise_schedule)

if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun."
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

pipe = load_pipeline(MODEL_ID, scheduler=BASELINE["scheduler"])
prompts = m2_prompts()
runner = SweepRunner(pipe, prompts, OUT, model_id=MODEL_ID, size=SIZE)
print("Default DPM++ timestep spacing:", runner.default_spacing)
prompts

## 1. Integration: CLIP text encoder → diffusion U-Net

```
caption ─► CLIP tokenizer ─► CLIP text encoder ─► (77 × 768) token embeddings
                                                        │ cross-attention (every U-Net block)
random latent (4×64×64) ─► U-Net predicts noise ◄───────┘
        ▲                        │ classifier-free guidance: ε = ε_uncond + w · (ε_text − ε_uncond)
        └── scheduler step ◄─────┘ (noise schedule: which timesteps, how much noise to remove)
                 │ after N steps
                 ▼
            VAE decoder ─► 512 × 512 image
```

`src/denoise.py` writes this loop out by hand. Below we check that it reproduces the pipeline's
image exactly (same prompt, seed and settings), which shows where the text, the guidance scale
and the noise schedule enter the model.

In [ ]:
emb = encode_text(pipe, prompts.prompt[:2])
print("Text encoder:", type(pipe.text_encoder).__name__, "| U-Net:", type(pipe.unet).__name__)
print("Prompt embeddings:", tuple(emb.shape), "= (prompts, tokens, features)")
print("U-Net cross-attention dim:", pipe.unet.config.cross_attention_dim, "(must match the 768 features)")

In [ ]:
p0, s0 = prompts.prompt[0], int(prompts.seed[0])
pipe.scheduler = make_scheduler("dpm++", runner.base_config, timestep_spacing=runner.default_spacing)

img_manual, ts = manual_generate(pipe, p0, guidance_scale=BASELINE["guidance_scale"],
                                 num_steps=BASELINE["num_steps"], seed=s0, size=SIZE, return_timesteps=True)
img_pipe = generate(pipe, prompts=[p0], guidance_scale=BASELINE["guidance_scale"],
                    num_steps=BASELINE["num_steps"], seed=s0, size=SIZE)[0][0]

diff = np.abs(np.asarray(img_manual, dtype=int) - np.asarray(img_pipe, dtype=int)).mean()
print(f"Mean pixel difference, manual loop vs pipeline: {diff:.2f} / 255  (< 1 means the same image)")
print(f"Timesteps visited ({len(ts)}):", ts[:4], "...", ts[-3:])

fig, axes = plt.subplots(1, 2, figsize=(8, 4.4))
for ax, im, t in zip(axes, [img_manual, img_pipe], ["manual loop (src/denoise.py)", "StableDiffusionPipeline"]):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
fig.suptitle(p0, fontsize=10); fig.tight_layout()
fig.savefig(FIG / "manual_vs_pipeline.png", dpi=110, bbox_inches="tight"); plt.show()

**Noise schedule.** SD 1.5 was trained with a fixed *scaled-linear* beta schedule over 1000 steps (left).
At inference the sampler visits only N of those steps; the **timestep spacing** decides which ones (right).

In [ ]:
plot_noise_schedule(runner.base_config, num_steps=BASELINE["num_steps"], spacings=SPACINGS,
                    save_as=FIG / "noise_schedule.png")
plt.show()

## 2. Baseline conditional generation
DPM-Solver++, 30 steps, guidance w = 7.5 (the Milestone 1 settings), on all 10 prompts.
Every later sweep reuses these images for its baseline value instead of generating them again.

In [ ]:
runner.run("baseline", "baseline", **BASELINE)
runner.log_df.tail(len(prompts))[["prompt_id", "prompt", "seed", "seconds", "file"]]

## 3. Classifier-free guidance sweep
Same prompts, seeds, sampler (DPM++) and steps (30); only the guidance scale w changes.
w = 1 turns guidance off (the pipeline then skips the unconditional pass, so it is also faster).

In [ ]:
for w in GUIDANCE_VALUES:
    runner.run("guidance", f"w = {w:g}", scheduler="dpm++", guidance_scale=w, num_steps=BASELINE["num_steps"])

In [ ]:
# Optional backup: download what is finished so far (in case Colab disconnects)
# download_outputs()

## 4. Noise schedule sweeps

**4a. Sampler.** Each scheduler uses a different rule to step along the noise schedule.
All four use the same 30 steps, w = 7.5 and the same timestep spacing.
DDPM was designed for ~1000 small steps, so it is expected to stay noisy at 30.

In [ ]:
NAMES = {"ddpm": "DDPM", "ddim": "DDIM", "pndm": "PNDM", "dpm++": "DPM++"}
for s in SAMPLERS:
    runner.run("sampler", NAMES[s], scheduler=s, guidance_scale=BASELINE["guidance_scale"],
               num_steps=BASELINE["num_steps"])

**4b. Timestep spacing.** Same sampler (DPM++), 30 steps, w = 7.5; only which noise levels are visited changes.
"leading" never starts from pure noise (its first step is below t = 999), "trailing" does.

In [ ]:
for sp in SPACINGS:
    runner.run("spacing", sp, scheduler="dpm++", guidance_scale=BASELINE["guidance_scale"],
               num_steps=BASELINE["num_steps"], timestep_spacing=sp)

## 5. (Optional) Number of sampling steps
DPM++, w = 7.5. Set `RUN_STEPS_SWEEP = False` in the settings cell to skip.

In [ ]:
if RUN_STEPS_SWEEP:
    for n in STEP_VALUES:
        runner.run("steps", f"{n} steps", scheduler="dpm++", guidance_scale=BASELINE["guidance_scale"], num_steps=n)

## 6. Generation log and CLIP score
The generation log has one row per image: experiment, setting, prompt, seed, scheduler, timestep spacing,
guidance, steps, seconds per image, CLIP score and file name. `blank = True` would mean the safety checker
replaced the image with a black one. CLIP score = 100 × cosine(image, prompt); real COCO pairs score 25.4.

In [ ]:
from src.embeddings import ClipEncoder

enc = ClipEncoder(CLIP_ID)
log = add_clip_scores(runner.log_df, OUT, enc)
runner.save_log(log)                                   # outputs/milestone2/generation_log.csv

summary = summarize(log)
summary.to_csv(OUT / "summary.csv", index=False)       # one row per setting (table for the report)
info = run_info(pipe, dict(baseline=BASELINE, default_timestep_spacing=runner.default_spacing,
                           prompts=len(prompts), size=SIZE))
with open(OUT / "run_info.json", "w") as f:
    json.dump(info, f, indent=2)

print(json.dumps(info, indent=1))
print("Blank (safety-checker) images:", int(log.drop_duplicates("file").blank.sum()))
summary

## 7. Figures
Early samples (the 5–10 images with prompts required for the report), then one grid per sweep:
rows are prompts, columns are settings, and each row uses the same seed.

In [ ]:
make_gallery(log, OUT, "baseline", save_as=FIG / "early_samples.png"); plt.show()

In [ ]:
make_grid(log, OUT, "guidance", title="Classifier-free guidance (DPM++, 30 steps)",
          save_as=FIG / "grid_guidance.png"); plt.show()
plot_guidance_curve(log, save_as=FIG / "clip_vs_guidance.png"); plt.show()

In [ ]:
make_grid(log, OUT, "sampler", title="Sampler (30 steps, w = 7.5)", save_as=FIG / "grid_sampler.png"); plt.show()
plot_setting_bars(log, "sampler", "Sampler: alignment and speed", save_as=FIG / "bars_sampler.png"); plt.show()

make_grid(log, OUT, "spacing", title="Timestep spacing (DPM++, 30 steps, w = 7.5)",
          save_as=FIG / "grid_spacing.png"); plt.show()
plot_setting_bars(log, "spacing", "Timestep spacing: alignment and speed", save_as=FIG / "bars_spacing.png"); plt.show()

if RUN_STEPS_SWEEP:
    make_grid(log, OUT, "steps", title="Sampling steps (DPM++, w = 7.5)", save_as=FIG / "grid_steps.png"); plt.show()
    plot_setting_bars(log, "steps", "Sampling steps: alignment and speed", save_as=FIG / "bars_steps.png"); plt.show()

In [ ]:
# Compact grids for the report: pick 3-4 prompts that show the differences best (see prompts table above)
REPORT_PROMPTS = ["coco_0", "coco_2", "hw_0", "hw_3"]
for exp in ["guidance", "sampler", "spacing"]:
    make_grid(log, OUT, exp, prompt_ids=REPORT_PROMPTS, save_as=FIG / f"report_grid_{exp}.png"); plt.show()

## 8. Observations
*TODO: fill in after looking at the grids, plots and `summary` table. Use numbers from `summary`.*

**Baseline**
- Which prompts worked well or badly? Compare COCO captions with handwritten prompts (CLIP score).

**Classifier-free guidance**
- What happens at w = 1 (no guidance)? At which w do images start to look oversaturated or distorted?
- Where does the CLIP score peak, and does it keep rising with w?
- Which w gives the best balance between realism and prompt-following?

**Noise schedule**
- Sampler: which sampler gives clean images in 30 steps? Why is DDPM noisy at 30 steps?
- Timestep spacing: does "trailing" (starts from pure noise, t = 999) change brightness/contrast compared with "leading"?
- Speed: seconds per image for each setting.

**Steps (optional)**
- How few steps still give good images?

**Next steps (Milestone 3)**
- Run the chosen settings on the 1k COCO subset and compute FID and IS against the 5k COCO images.

## 9. Save results
In Colab, run the cell below to download `milestone2_outputs.zip`. Unzip it into the local repo's
`outputs/` folder (it creates `outputs/milestone2/`), then **File → Download → Download .ipynb** and replace
`notebooks/03_milestone2_generation.ipynb`. Commit both with GitHub Desktop.

In [ ]:
download_outputs()